In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("marketing.db")
cursor = conn.cursor()

In [2]:
pd.read_sql(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
""",
conn
)

,name
0,ad_spend
1,web_logs
2,crm_conversions


In [3]:
mkdir sql

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("marketing.db")
cursor = conn.cursor()

# Create customer_journey Table

In [6]:
customer_journey_sql = """
DROP TABLE IF EXISTS customer_journey;

CREATE TABLE customer_journey AS
SELECT
    w.user_id,
    w.session_id,
    w.session_date,
    w.traffic_source,
    w.utm_source,
    w.utm_medium,
    w.utm_campaign,
    c.conversion_id,
    c.conversion_date,
    c.product,
    c.deal_value,
    c.conversion_status
FROM web_logs w
INNER JOIN crm_conversions c
ON w.user_id = c.user_id;
"""

cursor.executescript(customer_journey_sql)
conn.commit()

In [7]:
pd.read_sql("""
SELECT COUNT(*)
FROM customer_journey
""", conn)

,COUNT(*)
0,28445


# Create ordered_journey Table

In [8]:
ordered_sql = """
DROP TABLE IF EXISTS ordered_journey;

CREATE TABLE ordered_journey AS
SELECT
    user_id,
    traffic_source,
    session_date,
    ROW_NUMBER() OVER(
        PARTITION BY user_id
        ORDER BY session_date
    ) AS touch_order
FROM customer_journey;
"""

cursor.executescript(ordered_sql)
conn.commit()

In [9]:
pd.read_sql("""
SELECT *
FROM ordered_journey
LIMIT 10
""", conn)

,user_id,traffic_source,session_date,touch_order
0,1000,Facebook Ads,2025-01-02,1
1,1000,Facebook Ads,2025-01-02,2
2,1000,Facebook Ads,2025-01-02,3
3,1000,Instagram Ads,2025-01-08,4
4,1000,Instagram Ads,2025-01-08,5
5,1000,Instagram Ads,2025-01-08,6
6,1000,Organic Search,2025-11-23,7
7,1000,Organic Search,2025-11-23,8
8,1000,Organic Search,2025-11-23,9
9,1001,Google Ads,2024-01-09,1


# First-Touch Attribution

In [11]:
first_touch_sql = """
DROP TABLE IF EXISTS first_touch_attribution;

CREATE TABLE first_touch_attribution AS
SELECT
    cj.user_id,
    cj.conversion_id,
    cj.deal_value,
    oj.traffic_source AS attributed_channel
FROM customer_journey cj
JOIN ordered_journey oj
    ON cj.user_id = oj.user_id
WHERE oj.touch_order = 1;
"""

cursor.executescript(first_touch_sql)
conn.commit()

In [12]:
pd.read_sql("""
SELECT *
FROM first_touch_attribution
LIMIT 10
""", conn)

,user_id,conversion_id,deal_value,attributed_channel
0,1000,C001962,18327.73,Facebook Ads
1,1000,C001962,18327.73,Facebook Ads
2,1000,C001962,18327.73,Facebook Ads
3,1000,C004837,16180.06,Facebook Ads
4,1000,C004837,16180.06,Facebook Ads
5,1000,C004837,16180.06,Facebook Ads
6,1000,C009984,26924.90,Facebook Ads
7,1000,C009984,26924.90,Facebook Ads
8,1000,C009984,26924.90,Facebook Ads
9,1001,C000886,13793.37,Google Ads


# Last-Touch Attribution

In [21]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
AND name='last_touch_attribution'
""", conn)

,name
0,last_touch_attribution


In [27]:
pd.read_sql("""
SELECT COUNT(*) AS total_rows
FROM last_touch_attribution
""", conn)

,total_rows
0,28445


In [28]:
pd.read_sql("""
SELECT *
FROM last_touch_attribution
LIMIT 10
""", conn)

,user_id,conversion_id,deal_value,attributed_channel
0,1000,C001962,18327.73,Organic Search
1,1000,C001962,18327.73,Organic Search
2,1000,C001962,18327.73,Organic Search
3,1000,C004837,16180.06,Organic Search
4,1000,C004837,16180.06,Organic Search
5,1000,C004837,16180.06,Organic Search
6,1000,C009984,26924.90,Organic Search
7,1000,C009984,26924.90,Organic Search
8,1000,C009984,26924.90,Organic Search
9,1001,C000886,13793.37,LinkedIn Ads


# Linear Attribution

In [7]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,ad_spend
1,web_logs
2,crm_conversions
3,customer_journey
4,ordered_journey
5,first_touch_attribution
6,last_touch_attribution


In [8]:
print(sqlite3.sqlite_version)

3.51.0


In [9]:
linear_sql = """
DROP TABLE IF EXISTS linear_attribution;

CREATE TABLE linear_attribution AS

SELECT
    user_id,
    traffic_source,
    touch_order,
    COUNT(*) OVER (
        PARTITION BY user_id
    ) AS total_touchpoints,
    1.0 / COUNT(*) OVER (
        PARTITION BY user_id
    ) AS attribution_weight

FROM ordered_journey;
"""

cursor.executescript(linear_sql)
conn.commit()

print("Linear Attribution Created")

Linear Attribution Created


In [10]:
pd.read_sql("""
SELECT *
FROM linear_attribution
LIMIT 10
""", conn)

,user_id,traffic_source,touch_order,total_touchpoints,attribution_weight
0,1000,Facebook Ads,1,9,0.111111
1,1000,Facebook Ads,2,9,0.111111
2,1000,Facebook Ads,3,9,0.111111
3,1000,Instagram Ads,4,9,0.111111
4,1000,Instagram Ads,5,9,0.111111
5,1000,Instagram Ads,6,9,0.111111
6,1000,Organic Search,7,9,0.111111
7,1000,Organic Search,8,9,0.111111
8,1000,Organic Search,9,9,0.111111
9,1001,Google Ads,1,5,0.200000


In [11]:
pd.read_sql("""
SELECT COUNT(*) AS total_rows
FROM linear_attribution
""", conn)

,total_rows
0,28445


# KPI Calculations

# Total Spend

In [12]:
pd.read_sql("""
SELECT
    SUM(ad_spend) AS total_spend
FROM ad_spend
""", conn)

,total_spend
0,50905965.59


# CPC

In [13]:
pd.read_sql("""
SELECT
    SUM(ad_spend)*1.0/SUM(clicks) AS CPC
FROM ad_spend
""", conn)

,CPC
0,0.507399


# CTR

In [14]:
pd.read_sql("""
SELECT
    (
        SELECT SUM(ad_spend)
        FROM ad_spend
    )*1.0/
    COUNT(DISTINCT conversion_id) AS CAC
FROM customer_journey
""", conn)

,CAC
0,5394.867061


# ROAS

In [15]:
pd.read_sql("""
SELECT
    SUM(deal_value)*1.0/
    (
        SELECT SUM(ad_spend)
        FROM ad_spend
    ) AS ROAS
FROM customer_journey
""", conn)

,ROAS
0,13.970697


# ROI

In [16]:
pd.read_sql("""
SELECT
(
    SUM(deal_value)
    -
    (
        SELECT SUM(ad_spend)
        FROM ad_spend
    )
)
*100.0/
(
    SELECT SUM(ad_spend)
    FROM ad_spend
) AS ROI
FROM customer_journey
""", conn)

,ROI
0,1297.069727


In [17]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,ad_spend
1,web_logs
2,crm_conversions
3,customer_journey
4,ordered_journey
5,first_touch_attribution
6,last_touch_attribution
7,linear_attribution


# Attribution Summary

In [18]:
first_summary = pd.read_sql("""
SELECT
    attributed_channel,
    COUNT(*) AS conversions,
    SUM(deal_value) AS revenue
FROM first_touch_attribution
GROUP BY attributed_channel
ORDER BY revenue DESC
""", conn)

first_summary

,attributed_channel,conversions,revenue
0,Email,4286,1.086923e+08
1,Google Ads,4179,1.051517e+08
2,Instagram Ads,4108,1.049492e+08
3,Facebook Ads,4175,1.020457e+08
4,YouTube Ads,4037,1.001315e+08
5,Organic Search,3774,9.698317e+07
6,LinkedIn Ads,3886,9.323825e+07


In [19]:
last_summary = pd.read_sql("""
SELECT
    attributed_channel,
    COUNT(*) AS conversions,
    SUM(deal_value) AS revenue
FROM last_touch_attribution
GROUP BY attributed_channel
ORDER BY revenue DESC
""", conn)

last_summary

,attributed_channel,conversions,revenue
0,Google Ads,4547,1.117807e+08
1,YouTube Ads,4186,1.057658e+08
2,Organic Search,4028,1.020393e+08
3,LinkedIn Ads,3950,1.007851e+08
4,Facebook Ads,3970,9.821348e+07
5,Instagram Ads,3894,9.637045e+07
6,Email,3870,9.623702e+07


In [20]:
linear_summary = pd.read_sql("""
SELECT
    traffic_source,
    COUNT(*) AS touchpoints,
    SUM(attribution_weight) AS attributed_conversions
FROM linear_attribution
GROUP BY traffic_source
ORDER BY attributed_conversions DESC
""", conn)

linear_summary

,traffic_source,touchpoints,attributed_conversions
0,YouTube Ads,4231,464.685426
1,Organic Search,3994,458.633730
2,Google Ads,4303,454.675866
3,Email,4112,454.629040
4,Instagram Ads,3995,445.895743
5,Facebook Ads,3882,421.310750
6,LinkedIn Ads,3928,412.169444
